# Capítulo 5: Aprendizaje por Refuerzo Profundo: Deep Q-Networks (DQN)

## Continuidad con el Capítulo 4: Del Tablero Discreto al Almacén Continuo

En el **Capítulo 4**, nuestro robot logístico **OptiBot** aprendió a navegar por un almacén modelado como una cuadrícula discreta de $3 \times 3$ casillas ($S_1, S_2, \dots, S_9$) utilizando una **Tabla Q**.

Sin embargo, en el mundo físico real, los robots **no se teletransportan entre casillas discretas**: se desplazan en un **plano continuo** gobernado por **coordenadas métricas $(x, y) \in [0.0, 3.0] \times [0.0, 3.0]$**.

### ¿Por qué la Tabla Q fracasa en el mundo continuo?
- Si la posición es métrica continua, OptiBot puede encontrarse en $(0.523, 2.481)$, $(0.524, 2.480)$, o en infinitas coordenadas intermedias.
- ¡Una tabla Q requeriría **infinitas filas**, colapsando la memoria y haciendo imposible visitar cada punto!
- Por eso recurrimos a una **Red Neuronal Profunda** como aproximador de función $Q(\mathbf{s}, a; \boldsymbol{\theta})$, capaz de **generalizar e interpolar** decisiones inteligentes para posiciones que jamás ha visitado.

En este cuaderno implementamos el flujo maestro de **Deep Q-Learning (DQL)** y **Double DQN** presentado en el Capítulo 5, conectando directamente con los **10 pasos fundamentales** y la arquitectura dual: la **Red Policy** ($\boldsymbol{\theta}_{\text{policy}}$, donde entrena el agente) y la **Red Target** ($\boldsymbol{\theta}_{\text{target}}$, copia congelada/suave para el blanco de Bellman).


In [ ]:
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Fijamos semillas para reproducibilidad
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

print(f"PyTorch versión: {torch.__version__}")

## 1. El Entorno: Almacén Continuo de OptiBot ($3.0 \times 3.0$ metros)

El recinto conserva exactamente la topología del almacén del Capítulo 4:
- **Espacio de coordenadas continuas:** $x \in [0.0, 3.0]$ (Oeste a Este) e $y \in [0.0, 3.0]$ (Sur a Norte).
- **Inicio ($S_1$):** Zona $[0.0, 1.0] \times [2.0, 3.0]$, centro en $(0.5, 2.5)$. OptiBot inicia con una pequeña variación sensorial continua.
- **Estantería Central Infranqueable ($S_5$):** Caja fija $[1.0, 2.0] \times [1.0, 2.0]$. Si el robot intenta penetrarla, choca, rebota a su posición previa y recibe $R = -1.0$.
- **Límites Perimetrales del Almacén:** Si el robot rebasa $[0.0, 3.0] \times [0.0, 3.0]$, cae fuera del recinto: penalización severa $R = -10.0$ y fin de episodio (*Game Over*).
- **Bahía de Entrega / Meta ($S_9$):** Zona $[2.0, 3.0] \times [0.0, 1.0]$, centro en $(2.5, 0.5)$. Otorga premio $R = +10.0$ y concluye con éxito el episodio.
- **Acciones discretas de desplazamiento:**
  - `0: Norte (^)` $(\Delta y = +paso)$
  - `1: Sur (v)` $(\Delta y = -paso)$
  - `2: Este (>)` $(\Delta x = +paso)$
  - `3: Oeste (<)` $(\Delta x = -paso)$
  con paso de desplazamiento métrico ($0.45\text{ m}$) y leve ruido físico continuo (derrape de ruedas).
- **Vector de estado continuo (4 dimensiones):**
$$\mathbf{s} = \left[ \frac{x}{3.0}, \; \frac{y}{3.0}, \; \frac{x_{\text{meta}} - x}{3.0}, \; \frac{y_{\text{meta}} - y}{3.0} \right]$$

In [ ]:
class AlmacenContinuoOptiBot:
    """
    Extensión continua del almacén 3x3 de OptiBot (Capítulo 4).
    Las posiciones ya no son números enteros de casilla, sino coordenadas continuas (x, y).
    """
    def __init__(self, paso=0.45):
        self.paso = paso
        self.meta_x = 2.5
        self.meta_y = 0.5
        self.reset()

    def reset(self):
        # Inicia cerca del centro de S1 (0.5, 2.5) con perturbación continua
        self.x = 0.5 + np.random.uniform(-0.10, 0.10)
        self.y = 2.5 + np.random.uniform(-0.10, 0.10)
        return self._obtener_estado()

    def _obtener_estado(self):
        # Coordenadas continuas y vector relativo hacia la meta normalizados a [0, 1]
        return np.array([
            self.x / 3.0,
            self.y / 3.0,
            (self.meta_x - self.x) / 3.0,
            (self.meta_y - self.y) / 3.0
        ], dtype=np.float32)

    def step(self, accion):
        dx, dy = 0.0, 0.0
        if accion == 0:    dy = self.paso   # Norte (^)
        elif accion == 1:  dy = -self.paso  # Sur (v)
        elif accion == 2:  dx = self.paso   # Este (>)
        elif accion == 3:  dx = -self.paso  # Oeste (<)

        # Ruido físico en los actuadores (fricción y derrape en el suelo del almacén)
        ruido_x = np.random.uniform(-0.02, 0.02)
        ruido_y = np.random.uniform(-0.02, 0.02)
        nx = self.x + dx + ruido_x
        ny = self.y + dy + ruido_y

        # 1. Caída fuera de los límites del almacén: Game Over (-10.0)
        if nx < 0.0 or nx > 3.0 or ny < 0.0 or ny > 3.0:
            return self._obtener_estado(), -10.0, True

        # 2. Choque contra el muro central (S5: [1.0, 2.0] x [1.0, 2.0]): rebote
        if 1.0 <= nx <= 2.0 and 1.0 <= ny <= 2.0:
            return self._obtener_estado(), -1.0, False

        # Movimiento válido: actualizamos coordenadas continuas
        self.x, self.y = nx, ny

        # 3. Llegada a la Bahía de Entrega (S9: [2.0, 3.0] x [0.0, 1.0]): Éxito (+10.0)
        if self.x >= 2.0 and self.y <= 1.0:
            return self._obtener_estado(), 10.0, True

        # 4. Paso ordinario por el pasillo (-1.0 por consumo de batería)
        return self._obtener_estado(), -1.0, False

print("Entorno AlmacenContinuoOptiBot definido correctamente.")

## 2. Red Neuronal Q: La Red Policy (Calculadora de Perillas Inteligentes)

En vez de una matriz con 9 filas, usamos un Perceptrón Multicapa (MLP) de dos capas ocultas (64 neuronas cada una con activaciones ReLU).
- **Paso 1 del Algoritmo:** Se inicializa la red con pesos y sesgos aleatorios pequeños (las perillas internas $\boldsymbol{\theta}_{\text{policy}}$).
- **Entrada:** 4 valores continuos de telemetría sensorial $\mathbf{s} = [x, y, \Delta x_{\text{meta}}, \Delta y_{\text{meta}}]$.
- **Salida:** 4 valores $Q(\mathbf{s}, a)$, uno para cada dirección motriz cardinal ($0:\uparrow, 1:\downarrow, 2:\rightarrow, 3:\leftarrow$).


In [ ]:
class RedQOptiBot(nn.Module):
    def __init__(self, dim_estado=4, n_acciones=4):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(dim_estado, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_acciones)
        )

    def forward(self, x):
        return self.red(x)

print("Arquitectura RedQOptiBot compilada.")

## 3. Buffer de Repetición de Experiencias (*Experience Replay* - Pasos 3a y 3b)

- **Paso 3a:** Almacena las tuplas $(\mathbf{s}_t, a_t, r_{t+1}, \mathbf{s}_{t+1}, \text{terminal}_{t+1})$ en una estructura de **cola circular FIFO** (tamaño máximo fijo).
- **Paso 3b:** Al entrenar, no usamos el último paso (evita el *olvido catastrófico*); extraemos **mini-lotes aleatorios** para romper la correlación temporal entre pasos sucesivos y estabilizar el gradiente.


In [ ]:
class BufferReplay:
    def __init__(self, capacidad=20000):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, s, a, r, s_sig, terminado):
        self.memoria.append((s, a, r, s_sig, float(terminado)))

    def muestrear_lote(self, batch_size=32):
        lote = random.sample(self.memoria, batch_size)
        s, a, r, s_sig, d = zip(*lote)
        return (
            torch.FloatTensor(np.array(s)),
            torch.LongTensor(a).unsqueeze(1),
            torch.FloatTensor(r).unsqueeze(1),
            torch.FloatTensor(np.array(s_sig)),
            torch.FloatTensor(d).unsqueeze(1)
        )

    def __len__(self):
        return len(self.memoria)

print("BufferReplay listo para operar.")

## 4. Agente DQN / Double DQN: Red Policy y Red Target (Pasos 1, 2, 4, 6, 7, 8 y 10)

Implementamos los componentes clave del algoritmo en PyTorch:
1. **Arquitectura Dual (Pasos 1 y 2):**
   - **Red Policy (Online):** Toma decisiones y ajusta sus pesos paso a paso.
   - **Red Target:** Copia para estabilizar el blanco de Bellman sin que se desplace.
2. **Double DQN (Paso 6):** La **Red Policy** selecciona la mejor acción futura ($\arg\max_{a'} Q(\mathbf{s}', a'; \boldsymbol{\theta}_{\text{policy}})$), mientras que la **Red Target** evalúa su valor de forma imparcial ($Q(\mathbf{s}', a^*; \boldsymbol{\theta}_{\text{target}})$), eliminando el sesgo sistemático de sobreestimación.
3. **Pérdida y Aislamiento de Acción (Pasos 7 y 8):** Usamos `.gather(1, acciones)` para calcular la pérdida exclusivamente sobre la acción ejecutada, dejando las demás con error cero (idéntico a la sustitución en el vector target).
4. **Actualización de la Red Target (Paso 10):** Se sincronizan periódicamente los parámetros (mediante actualización suave Polyak con $\tau = 0.01$ o reemplazo escalonado).


In [ ]:
class AgenteDQN:
    def __init__(self, dim_estado=4, n_acciones=4, lr=1e-3, gamma=0.95, tau=0.01):
        self.n_acciones = n_acciones
        self.gamma = gamma
        self.tau = tau
        self.epsilon = 1.0
        self.eps_min = 0.05
        self.decaimiento = 0.985

        # Paso 1: Red Policy (donde entrena y toma decisiones)
        self.red_online = RedQOptiBot(dim_estado, n_acciones)
        # Paso 2: Red Target (copia para fijar el objetivo de Bellman)
        self.red_target = RedQOptiBot(dim_estado, n_acciones)
        self.red_target.load_state_dict(self.red_online.state_dict())

        self.optimizador = optim.Adam(self.red_online.parameters(), lr=lr)
        self.criterio_perdida = nn.SmoothL1Loss()  # Pérdida de Huber (robusta ante colisiones)

    def seleccionar_accion(self, estado):
        # Paso 4: Propagación en la Red Policy y regla épsilon-greedy
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_acciones)  # Exploración
        with torch.no_grad():
            s_tensor = torch.FloatTensor(estado).unsqueeze(0)
            valores_q = self.red_online(s_tensor)
            return int(valores_q.argmax(dim=1).item())  # Explotación (arg max)

    def aprender_de_lote(self, buffer, batch_size=32):
        # Paso 3b: Extracción de lote descorrelacionado
        if len(buffer) < batch_size:
            return 0.0

        estados, acciones, recompensas, estados_sig, terminados = buffer.muestrear_lote(batch_size)

        with torch.no_grad():
            # Paso 6: Objetivo de Bellman con Double DQN
            # La Red Policy ELIGE la mejor acción en s'
            mejores_acciones = self.red_online(estados_sig).argmax(dim=1, keepdim=True)
            # La Red Target EVALÚA el valor Q de esa acción
            q_futuro = self.red_target(estados_sig).gather(1, mejores_acciones)
            # Blanco de Bellman: Q_target = r si terminal, r + gamma*q_futuro si no terminal
            objetivos = recompensas + self.gamma * (1.0 - terminados) * q_futuro

        # Paso 7 y 8: Predicción de la Red Policy para la acción tomada (gather)
        q_predichos = self.red_online(estados).gather(1, acciones)

        # Error y descenso de gradiente (solo ajusta perillas de la Red Policy)
        perdida = self.criterio_perdida(q_predichos, objetivos)
        self.optimizador.zero_grad()
        perdida.backward()
        self.optimizador.step()

        # Paso 10: Sincronización de parámetros hacia la Red Target (actualización Polyak)
        for p_t, p_o in zip(self.red_target.parameters(), self.red_online.parameters()):
            p_t.data.copy_(self.tau * p_o.data + (1.0 - self.tau) * p_t.data)

        # Reducción paulatina de la exploración (epsilon-decay)
        self.epsilon = max(self.eps_min, self.epsilon * self.decaimiento)
        return perdida.item()

print("Agente Double DQN inicializado con éxito (con Red Policy y Red Target sincronizadas).")


## 5. Entrenamiento en el Almacén Continuo (Paso 9: Bucle Interactivo)

Entrenamos a OptiBot durante 250 episodios interactuando en el entorno continuo ($3\text{ m} \times 3\text{ m}$):
- En cada paso, captura el estado métrico continuo (Paso 3).
- Decide la acción con la Red Policy (Paso 4).
- Guarda la experiencia en la cola FIFO del Experience Replay (Paso 3a).
- Extrae lotes aleatorios para entrenar la Red Policy hacia el blanco de Bellman fijado por la Red Target (Pasos 3b, 6, 7 y 8).
- Transfiere progresivamente el aprendizaje a la Red Target (Paso 10).


In [ ]:
env = AlmacenContinuoOptiBot(paso=0.45)
agente = AgenteDQN(dim_estado=4, n_acciones=4, lr=1e-3, gamma=0.95, tau=0.01)
buffer = BufferReplay(capacidad=20000)

num_episodios = 250
retornos = []
perdidas = []
exitos = 0

for ep in range(1, num_episodios + 1):
    estado = env.reset()
    retorno_ep = 0.0
    perdidas_ep = []

    for t in range(25):  # Máximo de pasos por episodio
        accion = agente.seleccionar_accion(estado)
        estado_sig, recompensa, terminado = env.step(accion)

        buffer.guardar(estado, accion, recompensa, estado_sig, terminado)
        loss = agente.aprender_de_lote(buffer, batch_size=32)
        if loss > 0:
            perdidas_ep.append(loss)

        estado = estado_sig
        retorno_ep += recompensa

        if terminado:
            if recompensa == 10.0:
                exitos += 1
            break

    retornos.append(retorno_ep)
    perdidas.append(np.mean(perdidas_ep) if perdidas_ep else 0.0)

    if ep % 25 == 0 or ep == 1:
        promedio_ultimos = np.mean(retornos[-25:])
        print(f"Episodio {ep:3d}/{num_episodios} | Retorno: {retorno_ep:6.2f} | Promedio (25 eps): {promedio_ultimos:6.2f} | Epsilon: {agente.epsilon:.3f}")

print(f"\nEntrenamiento finalizado con éxito. Éxitos totales en entrenamiento: {exitos}/{num_episodios}")

## 6. Evaluación y Visualización de la Trayectoria Continua en el Almacén

Probamos la política aprendida de OptiBot de forma puramente codiciosa (explotación sin ruido, $\varepsilon = 0$) y graficamos:
1. **Curva de Retorno:** Progreso del aprendizaje a lo largo de los episodios.
2. **Plano 2D Continuo del Almacén:** Representación visual del recinto mostrando el muelle de inicio ($S_1$), la estantería central impenetrable ($S_5$), la bahía de entrega ($S_9$) y la **trayectoria continua real** seguida por el robot esquivando el obstáculo.

In [ ]:
# Realizamos una prueba codiciosa final
agente.epsilon = 0.0
estado = env.reset()
trayectoria_x = [env.x]
trayectoria_y = [env.y]
acciones_tomadas = []
nombres_acciones = ["Norte (^)", "Sur (v)", "Este (>)", "Oeste (<)"]

for paso_t in range(20):
    accion = agente.seleccionar_accion(estado)
    estado, recompensa, terminado = env.step(accion)
    trayectoria_x.append(env.x)
    trayectoria_y.append(env.y)
    acciones_tomadas.append(nombres_acciones[accion])
    if terminado:
        break

print(f"Resultado de la prueba: {'¡META ALCANZADA!' if recompensa == 10.0 else 'No completada'}")
print(f"Número de pasos: {len(acciones_tomadas)}")
print(f"Secuencia de acciones: {' -> '.join(acciones_tomadas)}")

# ============================================================
# Gráficas de Resultados
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 1. Curva de aprendizaje
ax1.plot(retornos, color='#028090', alpha=0.35, label='Retorno por episodio')
media_movil = np.convolve(retornos, np.ones(15)/15, mode='valid')
ax1.plot(range(14, len(retornos)), media_movil, color='#004e64', lw=2.5, label='Media Móvil (15 eps)')
ax1.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax1.set_title('Evolución del Aprendizaje (DQN)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Episodio', fontsize=11)
ax1.set_ylabel('Retorno Acumulado', fontsize=11)
ax1.grid(True, alpha=0.25)
ax1.legend(loc='lower right')

# 2. Mapa 2D Continuo del Almacén
ax2.set_xlim(-0.1, 3.1)
ax2.set_ylim(-0.1, 3.1)
ax2.set_aspect('equal')

# Zonas del almacén (correspondencia con Capítulo 4)
# Inicio S1: [0, 1] x [2, 3]
ax2.fill_between([0, 1], 2, 3, color='#e0f2f1', label='Inicio (S1: Muelle)')
# Estantería central S5: [1, 2] x [1, 2]
ax2.fill_between([1, 2], 1, 2, color='#78909c', hatch='//', label='Estantería S5 (Muro)')
# Meta S9: [2, 3] x [0, 1]
ax2.fill_between([2, 3], 0, 1, color='#c8e6c9', label='Meta (S9: Bahía Entrega)')

# Cuadrícula de referencia continua (líneas guía)
for c in [1.0, 2.0]:
    ax2.axvline(c, color='gray', linestyle=':', alpha=0.4)
    ax2.axhline(c, color='gray', linestyle=':', alpha=0.4)

# Límites del almacén
ax2.plot([0, 3, 3, 0, 0], [0, 0, 3, 3, 0], 'k-', lw=2, label='Límites Almacén')

# Trayectoria continua de OptiBot
ax2.plot(trayectoria_x, trayectoria_y, color='#d81159', lw=2.5, marker='o', markersize=6, label='Trayectoria OptiBot')
ax2.scatter(trayectoria_x[0], trayectoria_y[0], color='#028090', s=130, zorder=5, label='Punto Partida (x0, y0)')
ax2.scatter(trayectoria_x[-1], trayectoria_y[-1], color='#2e7d32', s=140, marker='*', zorder=5, label='Llegada a Meta')

# Anotaciones de coordenadas
ax2.text(0.5, 2.5, 'S1 (Inicio)', ha='center', va='center', fontsize=9, fontweight='bold', color='#004d40')
ax2.text(1.5, 1.5, 'S5 (Muro)', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
ax2.text(2.5, 0.5, 'S9 (+10)', ha='center', va='center', fontsize=9, fontweight='bold', color='#1b5e20')

ax2.set_title('Navegación Continua de OptiBot en el Almacén', fontsize=13, fontweight='bold')
ax2.set_xlabel('Coordenada X (metros)', fontsize=11)
ax2.set_ylabel('Coordenada Y (metros)', fontsize=11)
ax2.grid(True, alpha=0.25)
ax2.legend(loc='center left', bbox_to_anchor=(1.03, 0.5), fontsize=9)

plt.tight_layout()
plt.show()

## 7. Conclusiones y Comparación Clave con el Capítulo 4

1. **Continuidad del Modelo:** El agente resolvió el mismo desafío que en el Capítulo 4 (navegar del muelle $S_1$ a la bahía de entrega $S_9$ esquivando el muro central $S_5$), pero ahora sobre un **espacio continuo con fricción y ruido físico**.
2. **Poder de Generalización:** OptiBot nunca comenzó dos episodios exactamente en el mismo punto decimal (debido al ruido aleatorio en la posición de inicio y durante los desplazamientos). La red neuronal aprendió un campo continuo de valores $Q(\mathbf{s}, a)$ que le permite decidir la acción correcta desde cualquier coordenada.
3. **Estabilidad Algorítmica:** La combinación de **Experience Replay** y **Double DQN con actualización suave Polyak** evitó la temida *Tríada Mortal*, permitiendo una convergencia rápida y confiable hacia la política óptima.